In [2]:
import pandas as pd
import numpy as np

customers = pd.read_csv("C:/Users/lenovo/Desktop/FinRisk-X/database/customers.csv")
loans = pd.read_csv("C:/Users/lenovo/Desktop/FinRisk-X/database/loans.csv")
transactions = pd.read_csv("C:/Users/lenovo/Desktop/FinRisk-X/database/transactions.csv")

transactions["timestamp"] = pd.to_datetime(
    transactions["timestamp"]
)

print("Customers:", customers.shape)
print("Loans:", loans.shape)
print("Transactions:", transactions.shape)

Customers: (10000, 13)
Loans: (18000, 14)
Transactions: (200000, 12)


In [3]:
transaction_features = (
    transactions
    .groupby("customer_id")
    .agg(
        total_transactions=("transaction_id", "count"),
        total_spending=("amount", "sum"),
        average_transaction=("amount", "mean"),
        transaction_std=("amount", "std"),
        maximum_transaction=("amount", "max"),
        fraudulent_transactions=("is_fraud", "sum")
    )
    .reset_index()
)

In [4]:
transaction_features.head()

,customer_id,total_transactions,total_spending,average_transaction,transaction_std,maximum_transaction,fraudulent_transactions
0,100001,22,69646.23,3165.737727,3647.497736,13351.43,0
1,100002,22,53501.76,2431.898182,3422.756006,13250.18,1
2,100003,10,27538.27,2753.827000,2485.266425,7667.48,1
3,100004,23,78723.02,3422.740000,3150.940791,10738.96,2
4,100005,16,27553.89,1722.118125,1308.229087,5056.43,0


In [5]:
transaction_features["transaction_frequency"] = (
    transaction_features["total_transactions"]
)

In [6]:
transaction_features["spending_volatility"] = (
    transaction_features["transaction_std"] /
    transaction_features["average_transaction"]
)

In [7]:
transaction_features["spending_volatility"] = (
    transaction_features["spending_volatility"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [8]:
transaction_features["fraud_ratio"] = (
    transaction_features["fraudulent_transactions"] /
    transaction_features["total_transactions"]
)

In [9]:
loan_features = (
    loans
    .groupby("customer_id")
    .agg(
        total_loans=("loan_id", "count"),
        total_loan_amount=("loan_amount", "sum"),
        average_interest_rate=("interest_rate", "mean"),
        average_loan_term=("loan_term_months", "mean"),
        total_monthly_payment=("monthly_payment", "sum"),
        defaulted_loans=("default", "sum")
    )
    .reset_index()
)

In [10]:
customer_profile = customers.merge(
    loan_features,
    on="customer_id",
    how="left"
)

customer_profile = customer_profile.merge(
    transaction_features,
    on="customer_id",
    how="left"
)

In [11]:
numeric_columns = customer_profile.select_dtypes(
    include=np.number
).columns

customer_profile[numeric_columns] = (
    customer_profile[numeric_columns]
    .fillna(0)
)

In [12]:
customer_profile.shape

(10000, 28)

In [13]:
customer_profile["loan_to_income"] = (
    customer_profile["total_loan_amount"] /
    customer_profile["annual_income"]
)

In [14]:
customer_profile["debt_to_income"] = (
    customer_profile["existing_debt"] /
    customer_profile["annual_income"]
)

In [15]:
monthly_income = (
    customer_profile["annual_income"] / 12
)

customer_profile["payment_to_income"] = (
    customer_profile["total_monthly_payment"] /
    monthly_income
)

In [16]:
customer_profile["savings_to_loan"] = (
    customer_profile["savings"] /
    customer_profile["total_loan_amount"]
)

In [17]:
ratio_columns = [
    "loan_to_income",
    "debt_to_income",
    "payment_to_income",
    "savings_to_loan"
]

customer_profile[ratio_columns] = (
    customer_profile[ratio_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [20]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

health_features = [
    "credit_score",
    "debt_to_income",
    "payment_to_income",
    "savings_to_loan",
    "spending_volatility"
]

# Make sure required columns exist
missing = [
    col for col in health_features
    if col not in customer_profile.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

# Clean values
customer_profile[health_features] = (
    customer_profile[health_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# Scale features
scaler = MinMaxScaler()

scaled = scaler.fit_transform(
    customer_profile[health_features]
)

scaled_df = pd.DataFrame(
    scaled,
    columns=health_features,
    index=customer_profile.index
)

# Financial health score
customer_profile["financial_health_score"] = (
    0.30 * scaled_df["credit_score"]
    + 0.20 * (1 - scaled_df["debt_to_income"])
    + 0.20 * (1 - scaled_df["payment_to_income"])
    + 0.20 * scaled_df["savings_to_loan"]
    + 0.10 * (1 - scaled_df["spending_volatility"])
) * 100

customer_profile[
    [
        "customer_id",
        "financial_health_score"
    ]
].head()

,customer_id,financial_health_score
0,100001,64.975851
1,100002,65.109057
2,100003,55.265966
3,100004,70.999953
4,100005,56.226856


In [21]:
customer_profile["financial_health_category"] = pd.cut(
    customer_profile["financial_health_score"],
    bins=[-np.inf, 35, 65, np.inf],
    labels=[
        "High Risk",
        "Moderate",
        "Healthy"
    ]
)

customer_profile[
    "financial_health_category"
].value_counts()

financial_health_category
Moderate     6327
Healthy      3673
High Risk       0
Name: count, dtype: int64

In [24]:
customer_profile.to_csv(
    "C:/Users/lenovo/Desktop/FinRisk-X/database/customer_financial_profile.csv",
    index=False
)